In [1]:
import os
import glob
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ultralytics import YOLO
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

C:\Users\End-user\AppData\Roaming\Python\Python39\site-packages\matplotlib\projections\__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


In [2]:
DATASET_PATH = r"C:\Users\End-user\Desktop\Eman Mehmood 1\2nd experiment\Pods Detection.v5i.yolo26"

TEST_IMAGES = os.path.join(DATASET_PATH, "test", "images")
TEST_LABELS = os.path.join(DATASET_PATH, "test", "labels")

BEST_MODEL = r"C:\Users\End-user\Desktop\Eman Mehmood 1\2nd experiment\yolov26s\model testing\best.pt"

print("Test images:", TEST_IMAGES)
print("Test labels:", TEST_LABELS)
print("Best model:", BEST_MODEL)

print("Images folder exists:", os.path.exists(TEST_IMAGES))
print("Labels folder exists:", os.path.exists(TEST_LABELS))
print("Model exists:", os.path.exists(BEST_MODEL))

Test images: C:\Users\End-user\Desktop\Eman Mehmood 1\2nd experiment\Pods Detection.v5i.yolo26\test\images
Test labels: C:\Users\End-user\Desktop\Eman Mehmood 1\2nd experiment\Pods Detection.v5i.yolo26\test\labels
Best model: C:\Users\End-user\Desktop\Eman Mehmood 1\2nd experiment\yolov26s\model testing\best.pt
Images folder exists: True
Labels folder exists: True
Model exists: True


In [3]:
model = YOLO(BEST_MODEL)

print("Model loaded successfully.")
print("Model:", BEST_MODEL)

Model loaded successfully.
Model: C:\Users\End-user\Desktop\Eman Mehmood 1\2nd experiment\yolov26s\model testing\best.pt


In [4]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))




PyTorch: 2.5.1+cu124
CUDA available: True
GPU: NVIDIA RTX A4000


In [5]:
image_files = sorted(
    glob.glob(os.path.join(TEST_IMAGES, "*.jpg")) +
    glob.glob(os.path.join(TEST_IMAGES, "*.jpeg")) +
    glob.glob(os.path.join(TEST_IMAGES, "*.png"))
)

print("Number of test images:", len(image_files))

Number of test images: 57


In [6]:
DATA_YAML = os.path.join(DATASET_PATH, "data.yaml")

metrics = model.val(
    data=DATA_YAML,
    split="test",
    imgsz=640,
    conf=0.001,
    device=0,
    plots=True
)

Ultralytics 8.4.115  Python-3.9.13 torch-2.5.1+cu124 CUDA:0 (NVIDIA RTX A4000, 16375MiB)
YOLO26s summary (fused): 122 layers, 9,465,567 parameters, 0 gradients, 20.8 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 299.541.2 MB/s, size: 44.7 KB)
val: Scanning C:\Users\End-user\Desktop\Eman Mehmood 1\2nd experiment\Pods Detection.v5i.yolo26\test\labels.cache... 57 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 57/57  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 1.8it/s 2.2s0.5ss
                   all         57       1045      0.963      0.935      0.976      0.713
Speed: 2.0ms preprocess, 6.8ms inference, 0.0ms loss, 3.3ms postprocess per image
Results saved to runs\detect\val


In [7]:
precision = float(metrics.box.mp)
recall = float(metrics.box.mr)
map50 = float(metrics.box.map50)
map50_95 = float(metrics.box.map)

f1 = 2 * (precision * recall) / (precision + recall)

print("===== YOLO26s TEST RESULTS =====")
print(f"Precision      : {precision:.4f}")
print(f"Recall         : {recall:.4f}")
print(f"F1-score       : {f1:.4f}")
print(f"mAP@50         : {map50:.4f}")
print(f"mAP@50:95      : {map50_95:.4f}")

===== YOLO26s TEST RESULTS =====
Precision      : 0.9626
Recall         : 0.9353
F1-score       : 0.9488
mAP@50         : 0.9756
mAP@50:95      : 0.7134


In [8]:
def get_ground_truth_count(image_path):
    image_name = os.path.splitext(os.path.basename(image_path))[0]
    label_path = os.path.join(TEST_LABELS, image_name + ".txt")

    if not os.path.exists(label_path):
        return None

    with open(label_path, "r") as f:
        lines = [line.strip() for line in f.readlines() if line.strip()]

    return len(lines)

In [9]:
results_table = []

for image_path in image_files:

    start_time = time.perf_counter()

    results = model.predict(
        source=image_path,
        conf=0.7,
        imgsz=640,
        device=0,
        verbose=False
    )

    elapsed = time.perf_counter() - start_time

    result = results[0]

    predicted_count = len(result.boxes)
    ground_truth_count = get_ground_truth_count(image_path)

    absolute_error = None

    if ground_truth_count is not None:
        absolute_error = abs(
            ground_truth_count - predicted_count
        )

    results_table.append({
        "image": os.path.basename(image_path),
        "ground_truth_count": ground_truth_count,
        "predicted_count": predicted_count,
        "absolute_error": absolute_error,
        "inference_time_sec": elapsed,
        "inference_time_ms": elapsed * 1000
    })

results_df = pd.DataFrame(results_table)

results_df.head()

,image,ground_truth_count,predicted_count,absolute_error,inference_time_sec,inference_time_ms
0,IMG_0025_jpeg.rf.28ff0e1f3fbf3bac5f89b1eac2f3c...,17,13,4,0.142417,142.4172
1,IMG_0059_jpeg.rf.795a0843311a0044c830e6e571790...,21,19,2,0.061257,61.2572
2,IMG_0062_jpeg.rf.df5c821598de499c270156c06bb08...,19,18,1,0.060285,60.2852
3,IMG_0073_jpeg.rf.ba638fbf170c116a5e13dcdc08b8c...,5,5,0,0.060504,60.5037
4,IMG_0076_jpeg.rf.99bdfe62fe1ccec83e650c3ef3600...,11,9,2,0.038942,38.9417


In [10]:
valid_df = results_df.dropna(
    subset=["ground_truth_count", "predicted_count"]
).copy()

y_true = valid_df["ground_truth_count"].values
y_pred = valid_df["predicted_count"].values

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)

nonzero_mask = y_true != 0

mape = np.mean(
    np.abs(
        (y_true[nonzero_mask] - y_pred[nonzero_mask])
        / y_true[nonzero_mask]
    )
) * 100

print("===== SOYBEAN POD COUNTING RESULTS =====")
print(f"Test images : {len(valid_df)}")
print(f"MAE         : {mae:.4f} pods/image")
print(f"RMSE        : {rmse:.4f} pods/image")
print(f"MAPE        : {mape:.2f}%")
print(f"R²          : {r2:.4f}")

===== SOYBEAN POD COUNTING RESULTS =====
Test images : 57
MAE         : 2.1404 pods/image
RMSE        : 3.0493 pods/image
MAPE        : 12.00%
R²          : 0.9086


In [11]:
OUTPUT_CSV = "YOLO26s_soybean_pod_counting_results.csv"

results_df.to_csv(
    OUTPUT_CSV,
    index=False
)

print("Results saved to:", OUTPUT_CSV)

Results saved to: YOLO26s_soybean_pod_counting_results.csv


In [12]:
plt.figure(figsize=(7, 7))

plt.scatter(
    y_true,
    y_pred,
    alpha=0.7
)

min_value = min(y_true.min(), y_pred.min())
max_value = max(y_true.max(), y_pred.max())

plt.plot(
    [min_value, max_value],
    [min_value, max_value],
    linestyle="--"
)

plt.xlabel("Ground-truth pod count")
plt.ylabel("Predicted pod count")

plt.title(
    f"Soybean Pod Counting: Actual vs Predicted\n"
    f"MAE={mae:.2f}, RMSE={rmse:.2f}, R²={r2:.3f}"
)

plt.grid(True)
plt.tight_layout()

plt.savefig(
    "YOLO26s_actual_vs_predicted_pod_count.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

<Figure size 700x700 with 1 Axes>

In [13]:
PREDICTION_DIR = "YOLO26s_test_predictions"
os.makedirs(PREDICTION_DIR, exist_ok=True)

for image_path in image_files:

    results = model.predict(
        source=image_path,
        conf=0.7,
        imgsz=640,
        device=0,
        verbose=False
    )

    annotated = results[0].plot()

    output_path = os.path.join(
        PREDICTION_DIR,
        os.path.basename(image_path)
    )

    import cv2
    cv2.imwrite(output_path, annotated)

print("Annotated test images saved in:", PREDICTION_DIR)

Annotated test images saved in: YOLO26s_test_predictions


In [14]:
summary = pd.DataFrame({
    "Metric": [
        "Precision",
        "Recall",
        "F1-score",
        "mAP@50",
        "mAP@50:95",
        "Counting MAE",
        "Counting RMSE",
        "Counting MAPE",
        "Counting R²"
    ],
    "Value": [
        precision,
        recall,
        f1,
        map50,
        map50_95,
        mae,
        rmse,
        mape,
        r2
    ]
})

summary

,Metric,Value
0,Precision,0.962577
1,Recall,0.935326
2,F1-score,0.948756
3,mAP@50,0.975612
4,mAP@50:95,0.713378
5,Counting MAE,2.140351
6,Counting RMSE,3.049302
7,Counting MAPE,12.000171
8,Counting R²,0.908631
